In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
import ipywidgets as widgets
from pathlib import Path

print("Water Potability Dashboard - Libraries Ready")

Water Potability Dashboard - Libraries Ready


In [2]:
# 2. Load the dataset
# Keep water_potability.csv in the same folder as this notebook, or update DATA_PATH below.

possible_paths = [
    Path(r"C:\Users\vetri\Downloads\vetri padv a2\water_potability.csv"),
    Path(r"C:\Users\vetri\Downloads\vetri padv a2\water_potability(1).csv"),
    Path.cwd() / "water_potability.csv",
    Path.cwd() / "water_potability(1).csv",
    Path.home() / "Downloads" / "vetri padv a2" / "water_potability.csv",
    Path.home() / "Downloads" / "vetri padv a2" / "water_potability(1).csv",
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the CSV file. Put water_potability.csv in the same folder "
        "as this notebook, or edit DATA_PATH to the full file path."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded from: {DATA_PATH}")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Column names:", df.columns.tolist())
display(df.head())

Dataset loaded from: C:\Users\vetri\Downloads\vetri padv a2\water_potability.csv
Rows: 3276
Columns: 10
Column names: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [3]:
# 3. Clean and prepare the data
water_df = df.copy()

# Standardize column names to remove accidental leading/trailing spaces.
water_df.columns = water_df.columns.str.strip()

required_columns = [
    "ph", "Hardness", "Solids", "Chloramines", "Sulfate",
    "Conductivity", "Organic_carbon", "Trihalomethanes",
    "Turbidity", "Potability"
]
missing_columns = [col for col in required_columns if col not in water_df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

print("Missing values before cleaning:")
display(water_df.isnull().sum().to_frame("Missing Values"))

# Fill missing numeric values with each column's median.
for col in required_columns:
    water_df[col] = pd.to_numeric(water_df[col], errors="coerce")
    if water_df[col].isnull().any():
        water_df[col] = water_df[col].fillna(water_df[col].median())

water_df["Potability"] = water_df["Potability"].astype(int)

# This label is created here AND refreshed inside draw_dashboard() as a safeguard.
water_df["Water Status"] = water_df["Potability"].map({
    0: "Not Potable",
    1: "Potable"
})

print("Missing values after cleaning:", int(water_df[required_columns].isnull().sum().sum()))
print("Cleaned dataset shape:", water_df.shape)
display(water_df.head())

Missing values before cleaning:


,Missing Values
ph,491
Hardness,0
Solids,0
Chloramines,0
Sulfate,781
Conductivity,0
Organic_carbon,0
Trihalomethanes,162
Turbidity,0
Potability,0


Missing values after cleaning: 0
Cleaned dataset shape: (3276, 11)


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability,Water Status
0,7.036752,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0,Not Potable
1,3.716080,129.422921,18630.057858,6.635246,333.073546,592.885359,15.180013,56.329076,4.500656,0,Not Potable
2,8.099124,224.236259,19909.541732,9.275884,333.073546,418.606213,16.868637,66.420093,3.055934,0,Not Potable
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0,Not Potable
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0,Not Potable


In [4]:
BLUE = "#527DA8"
GREEN = "#26966F"
RED = "#DF5B61"

features = [
    "ph", "Hardness", "Solids", "Chloramines",
    "Sulfate", "Conductivity", "Organic_carbon",
    "Trihalomethanes", "Turbidity"
]

status_filter = widgets.Dropdown(
    options=[
        ("All Water Samples", "All"),
        ("Potable Only", "1"),
        ("Not Potable Only", "0")
    ],
    value="All",
    description="Status:",
    layout=widgets.Layout(width="48%")
)

feature_filter = widgets.Dropdown(
    options=features,
    value="ph",
    description="Parameter:",
    layout=widgets.Layout(width="48%")
)

out = widgets.Output()


def card(title, value, color):
    return f"""
    <div style="flex:1;min-width:160px;background:white;
    border-top:4px solid {color};padding:15px;
    border-radius:6px;box-shadow:0 1px 5px #ddd;">
        <div style="font-size:14px;color:#555">{title}</div>
        <div style="font-size:30px;color:#333;margin-top:10px">
        {value}</div>
    </div>
    """


def draw_dashboard(*args):
    data = water_df.copy()
    # Re-create the label on every refresh to avoid KeyError if an earlier cell was skipped.
    data["Water Status"] = data["Potability"].map({0: "Not Potable", 1: "Potable"})

    if status_filter.value != "All":
        data = data[
            data["Potability"] == int(status_filter.value)
        ]

    with out:
        out.clear_output(wait=True)

        if data.empty:
            print("No records available for this filter.")
            return

        total = len(data)
        potable = int((data["Potability"] == 1).sum())
        not_potable = int((data["Potability"] == 0).sum())
        potable_rate = potable / total * 100

        display(HTML(
            "<h2 style='font-family:Arial;font-weight:500'>"
            "Water Potability Analysis Dashboard</h2>"
            "<div style='display:flex;gap:12px;flex-wrap:wrap'>"
            + card("Total Samples", f"{total:,}", BLUE)
            + card("Potable Samples", f"{potable:,}", GREEN)
            + card("Not Potable Samples", f"{not_potable:,}", RED)
            + card("Potable Rate", f"{potable_rate:.2f}%", BLUE)
            + "</div>"
        ))

        # 1. Potability distribution
        counts = data["Water Status"].value_counts()

        fig = px.pie(
            names=counts.index,
            values=counts.values,
            hole=0.5,
            title="1. Potable vs Not Potable",
            color=counts.index,
            color_discrete_map={
                "Potable": GREEN,
                "Not Potable": RED
            }
        )
        fig.update_layout(template="plotly_white", height=350)
        display(fig)

        # 2. Selected water-quality parameter distribution
        feature = feature_filter.value

        fig = px.histogram(
            data,
            x=feature,
            color="Water Status",
            nbins=35,
            barmode="overlay",
            opacity=0.65,
            color_discrete_map={
                "Potable": GREEN,
                "Not Potable": RED
            },
            title=f"2. Distribution of {feature}"
        )
        fig.update_layout(template="plotly_white", height=380)
        display(fig)

        # 3. Average water-quality parameters
        averages = (
            data.groupby("Water Status")[features]
            .mean().T.reset_index()
            .rename(columns={"index": "Parameter"})
        )

        melted = averages.melt(
            id_vars="Parameter",
            var_name="Water Status",
            value_name="Average"
        )

        fig = px.bar(
            melted,
            x="Parameter",
            y="Average",
            color="Water Status",
            barmode="group",
            title="3. Average Parameters by Water Status",
            color_discrete_map={
                "Potable": GREEN,
                "Not Potable": RED
            }
        )
        fig.update_layout(
            template="plotly_white",
            height=420,
            xaxis_tickangle=-35
        )
        display(fig)

        # 4. pH vs Hardness
        fig = px.scatter(
            data,
            x="ph",
            y="Hardness",
            color="Water Status",
            opacity=0.65,
            color_discrete_map={
                "Potable": GREEN,
                "Not Potable": RED
            },
            title="4. pH vs Hardness"
        )
        fig.update_layout(template="plotly_white", height=380)
        display(fig)

        # 5. Correlation heatmap
        corr = data[features + ["Potability"]].corr()

        fig = px.imshow(
            corr,
            text_auto=".2f",
            color_continuous_scale="RdBu_r",
            zmin=-1,
            zmax=1,
            title="5. Water Quality Correlation Heatmap"
        )
        fig.update_layout(template="plotly_white", height=520)
        display(fig)

        # 6. Solids vs Conductivity
        fig = px.scatter(
            data,
            x="Solids",
            y="Conductivity",
            color="Water Status",
            opacity=0.65,
            color_discrete_map={
                "Potable": GREEN,
                "Not Potable": RED
            },
            title="6. Solids vs Conductivity"
        )
        fig.update_layout(template="plotly_white", height=380)
        display(fig)

        # 7. Box plot for selected parameter
        fig = px.box(
            data,
            x="Water Status",
            y=feature,
            color="Water Status",
            color_discrete_map={
                "Potable": GREEN,
                "Not Potable": RED
            },
            title=f"7. {feature} Comparison by Water Status"
        )
        fig.update_layout(template="plotly_white", height=380)
        display(fig)

        # 8. Chloramines distribution
        fig = px.histogram(
            data,
            x="Chloramines",
            color="Water Status",
            nbins=30,
            barmode="overlay",
            opacity=0.65,
            color_discrete_map={
                "Potable": GREEN,
                "Not Potable": RED
            },
            title="8. Chloramines Distribution"
        )
        fig.update_layout(template="plotly_white", height=380)
        display(fig)

        # 9. Sulfate distribution
        fig = px.histogram(
            data,
            x="Sulfate",
            color="Water Status",
            nbins=30,
            barmode="overlay",
            opacity=0.65,
            color_discrete_map={
                "Potable": GREEN,
                "Not Potable": RED
            },
            title="9. Sulfate Distribution"
        )
        fig.update_layout(template="plotly_white", height=380)
        display(fig)

        # 10. Trihalomethanes vs Turbidity
        fig = px.scatter(
            data,
            x="Trihalomethanes",
            y="Turbidity",
            color="Water Status",
            opacity=0.65,
            color_discrete_map={
                "Potable": GREEN,
                "Not Potable": RED
            },
            title="10. Trihalomethanes vs Turbidity"
        )
        fig.update_layout(template="plotly_white", height=380)
        display(fig)

        display(HTML(
            "<p style='color:#666;font-size:12px'>"
            "These charts show patterns in the supplied dataset. "
            "They do not certify that water is safe to drink.</p>"
        ))


status_filter.observe(draw_dashboard, names="value")
feature_filter.observe(draw_dashboard, names="value")

display(widgets.HBox([status_filter, feature_filter]))
display(out)
draw_dashboard()

Output()

In [5]:
# 5. Optional: quick data summary
summary = water_df.groupby("Water Status")[features].mean().round(2)
display(summary)

print("Dashboard ready. Use the Status and Parameter dropdowns above to explore the data.")

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
Water Status,,,,,,,,,
Not Potable,7.08,196.73,21777.49,7.09,334.20,426.73,14.36,66.32,3.97
Potable,7.07,195.80,22383.99,7.17,332.68,425.38,14.16,66.54,3.97


Dashboard ready. Use the Status and Parameter dropdowns above to explore the data.
